# MP Inversion Method Comparison

This notebook benchmarks all inversion methods on multiple synthetic benchmark laws.
It compares runtime and reconstruction quality side by side.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'mpdiff').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root (expected pyproject.toml and src/mpdiff).')
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mpdiff.config.schemas import MPForwardConfig, MPInverseConfig
from mpdiff.experiments.inversion_benchmark import benchmark_inverse_methods_from_population
from mpdiff.spectral.densities import (
    dirac_law,
    dirac_mixture_law,
    gamma_law,
    rescaled_beta_law,
)


In [ ]:
grid = np.linspace(0.01, 7.0, 500)
aspect_ratio = 0.25

forward_cfg = MPForwardConfig(
    aspect_ratio=aspect_ratio,
    grid_min=float(grid[0]),
    grid_max=float(grid[-1]),
    num_points=grid.size,
    eta=0.0035,
    tol=1e-9,
    max_iter=520,
    damping=0.72,
)

inverse_cfg = MPInverseConfig(
    method="optimization",
    compare_all_methods=True,
    n_support=55,
    support_min=0.05,
    support_max=5.5,
    eta=0.0035,
    tol=1e-6,
    max_iter=300,
    regularization=1e-3,
    optimizer_max_iter=120,
    forward_max_iter=240,
    forward_tol=1e-7,
)


In [ ]:
benchmark_laws = {
    "dirac": dirac_law(1.2).to_discrete(1),
    "dirac_mixture": dirac_mixture_law(
        values=np.array([0.5, 1.2, 2.4]),
        weights=np.array([0.25, 0.5, 0.25]),
    ).to_discrete(3),
    "gamma": gamma_law(shape=2.4, scale=0.55).to_discrete(450),
    "rescaled_beta": rescaled_beta_law(alpha=2.0, beta=5.5, scale=2.0, shift=0.2).to_discrete(450),
}

all_rows = []
results_by_problem = {}

for name, population in benchmark_laws.items():
    result = benchmark_inverse_methods_from_population(
        population=population,
        aspect_ratio=aspect_ratio,
        grid=grid,
        forward_settings=forward_cfg,
        inverse_settings=inverse_cfg,
        methods=None,
    )
    df = result.summary_table.copy()
    df["problem"] = name
    all_rows.append(df)
    results_by_problem[name] = result

benchmark_df = pd.concat(all_rows, ignore_index=True)
benchmark_df


In [ ]:
summary = (
    benchmark_df
    .groupby("method", as_index=False)[["runtime_seconds", "population_wasserstein_1", "reconstruction_l2"]]
    .mean()
    .sort_values(by=["population_wasserstein_1", "runtime_seconds"])
)
summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(summary["method"], summary["runtime_seconds"])
axes[0].set_title("Mean Runtime")
axes[0].set_ylabel("seconds")
axes[0].grid(alpha=0.2)

axes[1].bar(summary["method"], summary["population_wasserstein_1"])
axes[1].set_title("Mean Population W1")
axes[1].grid(alpha=0.2)

axes[2].bar(summary["method"], summary["reconstruction_l2"])
axes[2].set_title("Mean Reconstruction L2")
axes[2].grid(alpha=0.2)

for ax in axes:
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## Notes

- No single method dominates every benchmark law.
- Use this notebook to choose a default method for your workload based on both quality and runtime.
- Repeat with multiple seeds for stronger conclusions.
